# PySpark — Caching & Performance

PySpark uses **lazy evaluation** — no computation runs until an action is called.  
Every time you call an action on a DataFrame, PySpark re-reads and re-processes from the beginning.

**Problem:** Running `df.count()`, then `df.show()`, then `df.groupBy().agg()` = 3 full reads of the data.

**Solution:** Cache the DataFrame so it's computed once and stored in memory.

| Operation | What it does |
|-----------|-------------|
| `cache()` | Store in memory (same as `persist(MEMORY_AND_DISK)`) |
| `persist(level)` | Store with a specific storage level |
| `unpersist()` | Remove from cache |
| `repartition(n)` | Reshuffle into exactly N partitions (full shuffle) |
| `coalesce(n)` | Reduce partition count (no full shuffle) |

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum as spark_sum
from pyspark import StorageLevel
import time

spark = SparkSession.builder \
    .appName("CachingPerformance") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

# Generate a medium-sized DataFrame to illustrate caching benefit
df = spark.range(1_000_000).toDF("id") \
    .withColumn("value", (col("id") % 100).cast("double")) \
    .withColumn("category", (col("id") % 10).cast("string"))

print("DataFrame created with", df.count(), "rows")
df.printSchema()

## 1. The Problem — Without Caching, Data is Re-Computed Each Action

In [ ]:
# Without cache — every action re-reads and re-processes
t0 = time.time()
count1 = df.count()          # full scan 1
t1 = time.time()
avg_val = df.agg(avg("value")).collect()[0][0]   # full scan 2
t2 = time.time()

print(f"Without cache:")
print(f"  count()  : {count1:,}  [{t1-t0:.3f}s]")
print(f"  avg()    : {avg_val:.1f}  [{t2-t1:.3f}s]")
print(f"  Total    : {t2-t0:.3f}s  (data scanned twice)")

## 2. cache() — Store in Memory After First Computation

In [ ]:
# cache() marks the DataFrame to be stored after the first action runs
df.cache()   # or df.persist() — same thing

t0 = time.time()
count1 = df.count()   # first action — computes AND stores in cache
t1 = time.time()
avg_val = df.agg(avg("value")).collect()[0][0]   # reads from cache — fast!
t2 = time.time()
sum_val = df.groupBy("category").agg(spark_sum("value")).count()  # also from cache
t3 = time.time()

print("With cache:")
print(f"  count()       : {count1:,}  [{t1-t0:.3f}s]  (compute + store)")
print(f"  avg()         : {avg_val:.1f}  [{t2-t1:.3f}s]  (from cache)")
print(f"  groupBy+count : {sum_val}  [{t3-t2:.3f}s]  (from cache)")
print(f"  Total         : {t3-t0:.3f}s")

# Good practice: unpersist when done with this DataFrame
df.unpersist()
print("\nCache cleared.")
print("df.is_cached:", df.is_cached)   # False after unpersist

## 3. persist() — Choose Your Storage Level

`persist(StorageLevel.X)` gives you control over where to store the data.

| Level | Memory | Disk | Serialized | Replicas | Use when |
|-------|--------|------|------------|----------|----------|
| `MEMORY_ONLY` | ✓ | ✗ | ✗ | 1 | Data fits in RAM, fast |
| `MEMORY_AND_DISK` | ✓ | ✓ | ✗ | 1 | Default `cache()`, safe |
| `DISK_ONLY` | ✗ | ✓ | ✓ | 1 | Low RAM, can afford disk |
| `MEMORY_AND_DISK_2` | ✓ | ✓ | ✗ | 2 | High availability, 2 replicas |

In [ ]:
# Memory and disk — default (same as cache())
df.persist(StorageLevel.MEMORY_AND_DISK)
df.count()   # trigger caching
print("Persisted with MEMORY_AND_DISK")
print("is_cached:", df.is_cached)   # True
df.unpersist()

# Memory only — fastest but spills if RAM is full
# df.persist(StorageLevel.MEMORY_ONLY)

# Disk only — for data too large for RAM
# df.persist(StorageLevel.DISK_ONLY)

## 4. When to Cache

**Cache when:**
- A DataFrame is used in multiple actions (count + show + groupBy...)
- After an expensive transformation (join, complex aggregation) that's reused
- Iterative algorithms (ML training loops)

**Do NOT cache when:**
- The DataFrame is used only once
- The DataFrame is very large and RAM is limited
- You're in the middle of a transformation chain (not yet done building it)

In [ ]:
# Good caching pattern — expensive join result used multiple times
employees = spark.createDataFrame([
    (1, "Alice",   "Engineering", 95000),
    (2, "Bob",     "Marketing",   72000),
    (3, "Charlie", "Engineering", 110000),
    (4, "Diana",   "HR",          65000),
], ["id", "name", "dept", "salary"])

departments = spark.createDataFrame([
    ("Engineering", "NY", 500000),
    ("Marketing",   "CA", 300000),
    ("HR",          "TX", 150000),
], ["dept", "location", "budget"])

# Expensive join — cache the result if using it multiple times
enriched = employees.join(departments, on="dept", how="inner")
enriched.cache()

# Now multiple actions don't re-run the join
print("Total employees:", enriched.count())
enriched.groupBy("location").agg(avg("salary").alias("avg_salary")).show()
enriched.filter(col("salary") > 80000).show()

enriched.unpersist()

## 5. repartition() and coalesce() — Control Parallelism

The number of partitions controls how many tasks run in parallel.

| Method | Shuffle | Use case |
|--------|---------|----------|
| `repartition(n)` | Full shuffle | Increase OR decrease partitions, evenly distribute |
| `coalesce(n)` | Minimal / none | Only decrease partitions, avoids full shuffle |

In [ ]:
df_test = spark.range(100_000).toDF("id")

print("Default partitions:", df_test.rdd.getNumPartitions())

# repartition — full shuffle, even distribution
df_repartitioned = df_test.repartition(8)
print("After repartition(8):", df_repartitioned.rdd.getNumPartitions())

# coalesce — merge partitions, no shuffle (only decrease)
df_coalesced = df_test.coalesce(2)
print("After coalesce(2):", df_coalesced.rdd.getNumPartitions())

# When to use which:
print("\nrepartition() → increase or decrease, even distribution, full shuffle")
print("coalesce()    → decrease only, avoids shuffle, may be uneven")
print("coalesce(1)   → single output file (only for small data)")

# Repartition by column — data skew reduction
df_by_dept = employees.repartition(4, "dept")
print("Partitioned by dept:", df_by_dept.rdd.getNumPartitions())

## 6. spark.sql.shuffle.partitions — Key Configuration

The default number of shuffle partitions is **200** — far too many for small datasets, too few for very large ones.  
Always tune this for your dataset size.

In [ ]:
# Check current setting
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

# Change at runtime
spark.conf.set("spark.sql.shuffle.partitions", "8")
print("Updated to:", spark.conf.get("spark.sql.shuffle.partitions"))

# Or set at session creation
# spark = SparkSession.builder \
#     .config("spark.sql.shuffle.partitions", "8") \
#     .getOrCreate()

print("\nRule of thumb:")
print("  Small dev/test  : 4-8 partitions")
print("  Production      : 2-3x number of cores")
print("  Large datasets  : tune so each partition is ~128MB-256MB")

## Quick Reference

```python
from pyspark import StorageLevel

# Cache (default storage = MEMORY_AND_DISK)
df.cache()
df.count()        # first action triggers caching
df.unpersist()    # release when done

# Persist with specific level
df.persist(StorageLevel.MEMORY_ONLY)
df.persist(StorageLevel.MEMORY_AND_DISK)
df.persist(StorageLevel.DISK_ONLY)
df.unpersist()

# Check cache status
df.is_cached       # True / False

# Partitioning
df.rdd.getNumPartitions()   # current partition count
df.repartition(n)           # full shuffle, any direction
df.coalesce(n)              # merge only, no shuffle
df.repartition(n, "col")    # partition by column value

# Tune shuffle partitions (default 200 is too high for small data)
spark.conf.set("spark.sql.shuffle.partitions", "8")
```

## Interview Questions

1. What is lazy evaluation in PySpark? How does caching help?
2. What is the difference between `cache()` and `persist()`?
3. What does `unpersist()` do? What happens if you don't call it?
4. When should you NOT cache a DataFrame?
5. What is the difference between `repartition()` and `coalesce()`? Which causes a full shuffle?
6. What is `spark.sql.shuffle.partitions` and what is the default value? Why is 200 often too high?
7. What is data skew and how does `repartition(n, col)` help?

## Practice Exercises

**Easy:**
1. Cache a DataFrame and call `count()` twice — observe the timing difference.
2. Check the current partition count of a DataFrame. Repartition to 4 and verify.

**Medium:**
3. Create an expensive join, cache the result, and perform 3 different aggregations on it. Compare to uncached.
4. Write a DataFrame to a single CSV file using `coalesce(1)`.

**Advanced:**
5. Benchmark: read a large CSV, apply 3 transformations, then run 4 actions — once with caching and once without. Report the time difference.
6. Set `spark.sql.shuffle.partitions` to 4, run a `groupBy`, and check the number of output partitions.

In [ ]:
spark.stop()